# Pandas

**Subject:** Introduction to Problem Solving and Reasoning (IPSR)  
**Unit 1 (Python Basics), Topic 15**  
**How to use this notebook:** read each section, run every code cell (Shift + Enter), then change the examples and run them again. Try each exercise before you open its solution.

## 1. Learning goals

After this notebook you can:

* create a `Series` and a `DataFrame` and read their index, columns and dtypes
* look at data with `head`, `info` and `describe`
* select columns, rows by label (`loc`) and rows by position (`iloc`)
* filter rows with a condition, add columns and sort
* summarise groups with `groupby` and `agg`
* find and handle missing values
* read a CSV text into a table
* avoid the common traps: chained assignment, inclusive `loc` slices, `NaN != NaN`, dtype changes

## 2. Why pandas?

**pandas** is the standard Python library for tables of data, like a spreadsheet you control with code.
It has two main objects:

* **Series**: one column of values with a label (index) for each value
* **DataFrame**: a table, made of several Series that share one index

We import it with the usual short name `pd`. This notebook uses pandas 3.

In [1]:
import pandas as pd
import numpy as np
print(pd.__version__)

3.0.5


## 3. Series

A Series is a one-dimensional column. Each value has a label in the **index**. If you give no index, pandas uses `0, 1, 2, ...`.

In [2]:
marks = pd.Series([78, 64, 91, 55])
print(marks)
print("values:", marks.values)
print("index :", marks.index)
print("dtype :", marks.dtype)

0    78
1    64
2    91
3    55
dtype: int64
values: [78 64 91 55]
index : RangeIndex(start=0, stop=4, step=1)
dtype : int64


In [3]:
marks = pd.Series([78, 64, 91, 55], index=["maths", "physics", "python", "chem"], name="marks")
print(marks)
print(marks["python"])          # by label
print(marks.mean(), marks.max())
print(marks + 5)                # arithmetic works on every value

maths      78
physics    64
python     91
chem       55
Name: marks, dtype: int64
91
72.0 91
maths      83
physics    69
python     96
chem       60
Name: marks, dtype: int64


## 4. DataFrame

A DataFrame is a table. The easiest way to build one is from a dictionary: **keys become column names**, lists become the columns.

In [4]:
df = pd.DataFrame({
    "name":   ["Asha", "Ravi", "Meena", "Karan", "Sita", "Dev"],
    "branch": ["CSE", "ECE", "CSE", "ME", "ECE", "CSE"],
    "maths":  [78, 64, 91, 55, 82, 70],
    "python": [88, 59, 95, 48, 77, 66],
})
df

,name,branch,maths,python
0,Asha,CSE,78,88
1,Ravi,ECE,64,59
2,Meena,CSE,91,95
3,Karan,ME,55,48
4,Sita,ECE,82,77
5,Dev,CSE,70,66


In [5]:
print("shape  :", df.shape)          # (rows, columns)
print("columns:", list(df.columns))
print("index  :", df.index)
print(df.dtypes)

shape  : (6, 4)
columns: ['name', 'branch', 'maths', 'python']
index  : RangeIndex(start=0, stop=6, step=1)
name        str
branch      str
maths     int64
python    int64
dtype: object


In pandas 3, text columns have the dtype `str` (shown here as `str`). Number columns here are `int64`.

## 5. First look: `head`, `info`, `describe`

In [6]:
df.head(3)        # first 3 rows (default is 5); df.tail(3) gives the last rows

,name,branch,maths,python
0,Asha,CSE,78,88
1,Ravi,ECE,64,59
2,Meena,CSE,91,95


In [7]:
df.info()         # columns, non-null counts, dtypes

<class 'pandas.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   name    6 non-null      str  
 1   branch  6 non-null      str  
 2   maths   6 non-null      int64
 3   python  6 non-null      int64
dtypes: int64(2), str(2)
memory usage: 324.0 bytes


In [8]:
df.describe()     # count, mean, std, min, quartiles, max for number columns

,maths,python
count,6.000000,6.000000
mean,73.333333,72.166667
std,12.987173,17.837227
min,55.000000,48.000000
25%,65.500000,60.750000
50%,74.000000,71.500000
75%,81.000000,85.250000
max,91.000000,95.000000


## 6. Selecting columns

* `df["maths"]` gives a **Series**
* `df[["name", "maths"]]` (a list inside the brackets) gives a **DataFrame**

In [9]:
print(type(df["maths"]))
print(df["maths"].mean())
df[["name", "maths"]]

<class 'pandas.Series'>
73.33333333333333


,name,maths
0,Asha,78
1,Ravi,64
2,Meena,91
3,Karan,55
4,Sita,82
5,Dev,70


## 7. Selecting rows: `loc` and `iloc`

| | selects by | end of a slice |
|---|---|---|
| `df.loc[...]` | **label** (index value, column name) | **included** |
| `df.iloc[...]` | **position** (0, 1, 2, ...) | excluded (like normal Python) |

In [10]:
print(df.loc[2])                 # row with label 2
print()
print(df.loc[2, "name"])         # row label 2, column "name"
print(df.iloc[2, 0])             # row position 2, column position 0

name      Meena
branch      CSE
maths        91
python       95
Name: 2, dtype: object

Meena
Meena


In [11]:
df.iloc[0:3]                     # positions 0, 1, 2 (3 rows)

,name,branch,maths,python
0,Asha,CSE,78,88
1,Ravi,ECE,64,59
2,Meena,CSE,91,95


In [12]:
df.loc[0:3, ["name", "maths"]]   # labels 0, 1, 2 AND 3 (4 rows): the end is included

,name,maths
0,Asha,78
1,Ravi,64
2,Meena,91
3,Karan,55


We can also make a column the index, and then rows are found by name.

In [13]:
by_name = df.set_index("name")
print(by_name.loc["Meena", "python"])
by_name.loc["Ravi":"Sita"]       # label slice, "Sita" is included

95


,branch,maths,python
name,,,
Ravi,ECE,64,59
Meena,CSE,91,95
Karan,ME,55,48
Sita,ECE,82,77


## 8. Filtering rows with a condition

A comparison on a column gives a Series of `True` and `False` (a **boolean mask**). Put the mask inside `[ ]` to keep the `True` rows.
Join conditions with `&` (and), `|` (or), `~` (not). **Each condition needs its own brackets.**

In [14]:
mask = df["maths"] >= 75
print(mask)
df[mask]

0     True
1    False
2     True
3    False
4     True
5    False
Name: maths, dtype: bool


,name,branch,maths,python
0,Asha,CSE,78,88
2,Meena,CSE,91,95
4,Sita,ECE,82,77


In [15]:
df[(df["branch"] == "CSE") & (df["python"] > 70)]

,name,branch,maths,python
0,Asha,CSE,78,88
2,Meena,CSE,91,95


In [16]:
df[df["branch"].isin(["ECE", "ME"])]

,name,branch,maths,python
1,Ravi,ECE,64,59
3,Karan,ME,55,48
4,Sita,ECE,82,77


## 9. Adding columns and sorting

Assigning to a new column name creates the column. The calculation is done for every row at once.

In [17]:
df["total"] = df["maths"] + df["python"]
df["result"] = np.where(df["total"] >= 140, "pass", "fail")
df

,name,branch,maths,python,total,result
0,Asha,CSE,78,88,166,pass
1,Ravi,ECE,64,59,123,fail
2,Meena,CSE,91,95,186,pass
3,Karan,ME,55,48,103,fail
4,Sita,ECE,82,77,159,pass
5,Dev,CSE,70,66,136,fail


In [18]:
df.sort_values("total", ascending=False)       # returns a new, sorted table

,name,branch,maths,python,total,result
2,Meena,CSE,91,95,186,pass
0,Asha,CSE,78,88,166,pass
4,Sita,ECE,82,77,159,pass
5,Dev,CSE,70,66,136,fail
1,Ravi,ECE,64,59,123,fail
3,Karan,ME,55,48,103,fail


In [19]:
df.sort_values(["branch", "total"], ascending=[True, False])

,name,branch,maths,python,total,result
2,Meena,CSE,91,95,186,pass
0,Asha,CSE,78,88,166,pass
5,Dev,CSE,70,66,136,fail
4,Sita,ECE,82,77,159,pass
1,Ravi,ECE,64,59,123,fail
3,Karan,ME,55,48,103,fail


`sort_values` does not change `df`. Look at the first rows of `df` again: the order is the same as before. To keep a sorted table, assign it to a name.

In [20]:
df.head(2)

,name,branch,maths,python,total,result
0,Asha,CSE,78,88,166,pass
1,Ravi,ECE,64,59,123,fail


## 10. `groupby` and `agg`

`groupby` splits the rows into groups, then you summarise each group.

In [21]:
df.groupby("branch")["total"].mean()

branch
CSE    162.666667
ECE    141.000000
ME     103.000000
Name: total, dtype: float64

In [22]:
df.groupby("branch").agg(
    students=("name", "count"),
    avg_maths=("maths", "mean"),
    best_python=("python", "max"),
)

,students,avg_maths,best_python
branch,,,
CSE,3,79.666667,95
ECE,2,73.000000,77
ME,1,55.000000,48


## 11. Missing values

A missing value is shown as `NaN` ("not a number"). It comes from empty cells, or from a failed conversion.

* `isna()` marks the missing values
* `fillna(x)` replaces them with `x`
* `dropna()` removes rows that contain any missing value

In [23]:
scores = pd.DataFrame({
    "name":  ["Asha", "Ravi", "Meena", "Karan"],
    "quiz1": [8.0, 6.5, np.nan, 5.0],
    "quiz2": [9.0, np.nan, 7.5, np.nan],
})
print(scores)
print()
print(scores.isna())
print()
print(scores.isna().sum())      # missing values per column

    name  quiz1  quiz2
0   Asha    8.0    9.0
1   Ravi    6.5    NaN
2  Meena    NaN    7.5
3  Karan    5.0    NaN

    name  quiz1  quiz2
0  False  False  False
1  False  False   True
2  False   True  False
3  False  False   True

name     0
quiz1    1
quiz2    2
dtype: int64


In [24]:
print(scores.fillna(0))
print()
print(scores.fillna({"quiz1": scores["quiz1"].mean()}))   # fill one column with its mean
print()
print(scores.dropna())

    name  quiz1  quiz2
0   Asha    8.0    9.0
1   Ravi    6.5    0.0
2  Meena    0.0    7.5
3  Karan    5.0    0.0

    name  quiz1  quiz2
0   Asha    8.0    9.0
1   Ravi    6.5    NaN
2  Meena    6.5    7.5
3  Karan    5.0    NaN

   name  quiz1  quiz2
0  Asha    8.0    9.0


## 12. Reading a CSV

Real data usually comes from a file. `pd.read_csv` works with a file name, a web address, or any file-like object.
Here we use `io.StringIO` to treat a piece of text as if it were a file, so no download is needed.

In [25]:
import io
csv_text = '''city,month,temp_c,rain_mm
Bhopal,Jan,15,2
Bhopal,Jul,26,310
Delhi,Jan,13,19
Delhi,Jul,31,210
Chennai,Jan,25,24
Chennai,Jul,32,95
'''
weather = pd.read_csv(io.StringIO(csv_text))
print(weather)
print(weather.dtypes)

      city month  temp_c  rain_mm
0   Bhopal   Jan      15        2
1   Bhopal   Jul      26      310
2    Delhi   Jan      13       19
3    Delhi   Jul      31      210
4  Chennai   Jan      25       24
5  Chennai   Jul      32       95
city         str
month        str
temp_c     int64
rain_mm    int64
dtype: object


In [26]:
weather.groupby("month")[["temp_c", "rain_mm"]].mean()

,temp_c,rain_mm
month,,
Jan,17.666667,15.0
Jul,29.666667,205.0


In [27]:
# Writing is just as easy. Here we write to text instead of a file.
print(weather.head(2).to_csv(index=False))

city,month,temp_c,rain_mm
Bhopal,Jan,15,2
Bhopal,Jul,26,310



## 13. Edge cases

### 13.1 Chained assignment does not work (copy-on-write)
In pandas 3 every selection behaves like a new copy. So changing a result of `df[...]` does **not** change `df`.

In [28]:
t = pd.DataFrame({"a": [1, 2, 3], "b": [10, 20, 30]})

import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")      # pandas warns here; we hide it to keep the output short
    t[t["a"] > 1]["b"] = 0               # changes a temporary copy only
print(t)

   a   b
0  1  10
1  2  20
2  3  30


The correct way is one `loc` step that says rows and column together.

In [29]:
t.loc[t["a"] > 1, "b"] = 0
print(t)

   a   b
0  1  10
1  2   0
2  3   0


For the same reason, a new name made from a column is independent. Changing it does not touch the original.

In [30]:
col = t["a"]
col[0] = 100
print(col.tolist(), "vs original", t["a"].tolist())

[100, 2, 3] vs original [1, 2, 3]


### 13.2 `loc` slices include the end, `iloc` slices do not

In [31]:
s = pd.Series([10, 20, 30, 40, 50], index=list("abcde"))
print(s.loc["b":"d"].tolist())    # b, c, d
print(s.iloc[1:3].tolist())       # positions 1 and 2 only

[20, 30, 40]
[20, 30]


### 13.3 Integer index confusion
After filtering or sorting, the labels stay with their rows. So label `2` and position `2` can be different rows.

In [32]:
top = df.sort_values("total", ascending=False)[["name", "total"]]
print(top)
print()
print("loc[2] :", top.loc[2, "name"])    # the row labelled 2
print("iloc[2]:", top.iloc[2]["name"])   # the third row on the screen

    name  total
2  Meena    186
0   Asha    166
4   Sita    159
5    Dev    136
1   Ravi    123
3  Karan    103

loc[2] : Meena
iloc[2]: Sita


In [33]:
top = top.reset_index(drop=True)         # make a fresh 0, 1, 2 ... index
print(top.loc[2, "name"], top.iloc[2]["name"])

Sita Sita


### 13.4 `NaN` is not equal to anything, even itself
So you cannot find missing values with `==`. Use `isna()`.

In [34]:
print(np.nan == np.nan)
x = pd.Series([1.0, np.nan, 3.0])
print((x == np.nan).tolist())      # all False
print(x.isna().tolist())

False
[False, False, False]
[False, True, False]


### 13.5 Statistics skip `NaN` by default
`mean` ignores missing values. This is useful, but it changes the denominator. Two values, not three.

In [35]:
print(x.mean())                # (1 + 3) / 2
print(x.mean(skipna=False))    # nan
print(x.count(), len(x))       # count ignores NaN, len does not
print(x.fillna(0).mean())      # (1 + 0 + 3) / 3: a different answer

2.0
nan
2 3
1.3333333333333333


### 13.6 A missing value can change the dtype
Whole numbers cannot hold `NaN`, so the column becomes `float64`. Text columns with a missing value hold `NaN` too.

In [36]:
print(pd.Series([1, 2, 3]).dtype)
print(pd.Series([1, 2, None]).dtype)           # now float64
print(pd.Series([1, 2, None]).tolist())

# If you need whole numbers with gaps, use the nullable integer type 'Int64'
y = pd.Series([1, 2, None], dtype="Int64")
print(y.dtype)
print(y.tolist())

int64
float64
[1.0, 2.0, nan]
Int64
[1, 2, <NA>]


In [37]:
names = pd.Series(["Asha", None, "Ravi"])
print(names.dtype)          # text type
print(names.isna().tolist())

mixed = pd.Series([1, "two", 3.0])
print(mixed.dtype)          # object: a mixed bag, sums and means will fail
try:
    mixed.sum()
except TypeError as e:
    print("TypeError:", e)

str
[False, True, False]
object
TypeError: unsupported operand type(s) for +: 'int' and 'str'


### 13.7 Numbers stored as text
A column read as text cannot be averaged. Convert it with `pd.to_numeric`. Use `errors="coerce"` to turn bad entries into `NaN`.

In [38]:
raw = pd.read_csv(io.StringIO("id,score\n1,80\n2,absent\n3,70\n"))
print(raw.dtypes)
raw["score"] = pd.to_numeric(raw["score"], errors="coerce")
print(raw.dtypes)
print(raw["score"].mean())

id       int64
score      str
dtype: object
id         int64
score    float64
dtype: object
75.0


## 14. Quick summary

| Task | Code |
|---|---|
| Table from dict | `pd.DataFrame({"col": [..], ...})` |
| Size, types | `df.shape`, `df.dtypes`, `df.info()` |
| Quick look | `df.head()`, `df.describe()` |
| One column / several | `df["a"]`, `df[["a", "b"]]` |
| By label / by position | `df.loc[row, col]`, `df.iloc[row, col]` (`loc` includes the end) |
| Filter | `df[(cond1) & (cond2)]` |
| New column | `df["new"] = df["a"] + df["b"]` |
| Sort | `df.sort_values("a", ascending=False)` |
| Group | `df.groupby("g").agg(name=("col", "mean"))` |
| Missing | `isna()`, `fillna(x)`, `dropna()` |
| Read CSV | `pd.read_csv(path_or_StringIO)` |
| Set values safely | `df.loc[mask, "col"] = value` |

## 15. Exercises

Use this table in the exercises. Run the cell first.

In [39]:
shop = pd.DataFrame({
    "item":  ["pen", "book", "bag", "pencil", "ruler", "bottle"],
    "kind":  ["stationery", "stationery", "carry", "stationery", "stationery", "carry"],
    "price": [10, 120, 650, 5, 20, 250],
    "stock": [100, 40, 12, 300, 0, 25],
})
shop

,item,kind,price,stock
0,pen,stationery,10,100
1,book,stationery,120,40
2,bag,carry,650,12
3,pencil,stationery,5,300
4,ruler,stationery,20,0
5,bottle,carry,250,25


### Exercise 1

Print the shape of `shop`, the names of its columns and the dtype of the `price` column.

In [40]:
# Write your answer here

<details>
<summary><b>Show solution 1</b></summary>

```python
print(shop.shape)
print(list(shop.columns))
print(shop["price"].dtype)
```

</details>

### Exercise 2

Show only the `item` and `price` of the rows where the price is more than 100.

In [41]:
# Write your answer here

<details>
<summary><b>Show solution 2</b></summary>

You can also write `shop.loc[shop["price"] > 100, ["item", "price"]]`.

```python
shop[shop["price"] > 100][["item", "price"]]
```

</details>

### Exercise 3

Add a column `value` that is `price * stock`. Then print the two rows with the highest `value`.

*Hint:* `sort_values(..., ascending=False).head(2)`.

In [42]:
# Write your answer here

<details>
<summary><b>Show solution 3</b></summary>

Bag: 650 x 12 = 7800. Book: 120 x 40 = 4800.

```python
shop["value"] = shop["price"] * shop["stock"]
print(shop.sort_values("value", ascending=False).head(2))
```

</details>

### Exercise 4

For each `kind`, show the number of items and the average price (use `groupby` and `agg`).

In [43]:
# Write your answer here

<details>
<summary><b>Show solution 4</b></summary>

stationery: 4 items, average (10 + 120 + 5 + 20) / 4 = 38.75. carry: 2 items, average 450.

```python
shop.groupby("kind").agg(items=("item", "count"), avg_price=("price", "mean"))
```

</details>

### Exercise 5

Use `iloc` to print the last two rows of `shop`, and use `loc` to print the `price` of the row with label 1. In one comment, say what `shop.loc[1:2]` would return and how many rows.

In [44]:
# Write your answer here

<details>
<summary><b>Show solution 5</b></summary>

```python
print(shop.iloc[-2:])
print(shop.loc[1, "price"])
# shop.loc[1:2] returns 2 rows (labels 1 and 2) because loc includes the end
print(len(shop.loc[1:2]))
```

</details>

### Exercise 6

Read this CSV text with `read_csv` and `io.StringIO`. Fill the missing `marks` with the column mean and print the result.

```
name,marks
Asha,80
Ravi,
Meena,60
```

In [45]:
# Write your answer here

<details>
<summary><b>Show solution 6</b></summary>

The mean of 80 and 60 is 70 (the missing value is skipped), so Ravi gets 70.0.

```python
import io
text = "name,marks\nAsha,80\nRavi,\nMeena,60\n"
d = pd.read_csv(io.StringIO(text))
d["marks"] = d["marks"].fillna(d["marks"].mean())
print(d)
```

</details>

### Exercise 7

**Edge case.** Set the `stock` of every item that has `stock == 0` to `10`, correctly, so that `shop` itself changes. Then print `shop["stock"].tolist()`.

In [46]:
# Write your answer here

<details>
<summary><b>Show solution 7</b></summary>

`shop[shop["stock"] == 0]["stock"] = 10` would change a temporary copy and leave `shop` as it was.

```python
shop.loc[shop["stock"] == 0, "stock"] = 10
print(shop["stock"].tolist())
```

</details>

## 16. Where to read more

* Software Carpentry, *Pandas DataFrames*: https://swcarpentry.github.io/python-novice-gapminder/08-data-frames.html
* Python Data Science Handbook, *Introducing Pandas Objects*: https://jakevdp.github.io/PythonDataScienceHandbook/03.01-introducing-pandas-objects.html
* pandas documentation, *10 minutes to pandas*: https://pandas.pydata.org/docs/user_guide/10min.html